# Data Wrangling

Load & transform NBA dataset

Ensure the two datasets are located in folder called data (or update load filepath)

### Import & Load Data

In [ ]:
# Install polars if not already installed
%pip install polars

# Imports
import pandas as pd
import polars as pl
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder


In [ ]:
# Load raw csv data
df_shots = pl.read_csv("shotdetail_2024.csv", try_parse_dates=True)
df_cdnnba = pl.read_csv("cdnnba_2024.csv", try_parse_dates=True)

# Merge cdnnba & shots dataframes
df_merged = df_shots.join(df_cdnnba, how='left', left_on=['GAME_ID', 'GAME_EVENT_ID'], right_on=['gameId', 'actionNumber'])
df_merged = df_merged.with_columns(pl.concat_str([pl.col('GAME_ID'), pl.col('GAME_EVENT_ID')], separator="_").alias('Record_Id'))


### Exploration

In [ ]:
df_merged.dtypes

In [ ]:
df_merged.describe()

## Data Cleaning

In [ ]:
# Convert data types
df_merged = df_merged.with_columns(
    pl.col('GAME_DATE').cast(pl.String).str.to_datetime(format="%Y%m%d").alias('game_date'),
    pl.when(pl.col('HTM') == pl.col('teamTricode')).then(1).otherwise(0).alias('isHomeTeam'),
    pl.duration(minutes=pl.col('MINUTES_REMAINING'), seconds=pl.col('SECONDS_REMAINING')).alias('time_remaining'),
    (pl.col('scoreHome') - pl.col('scoreAway')).alias('score_margin_after'),
    pl.when(pl.col('SHOT_MADE_FLAG') == 1).then(
        pl.when(pl.col('actionType') == "3pt").then(pl.lit(3)).otherwise(pl.lit(2))
        ).otherwise(pl.lit(0)).alias('score_change'), 
    pl.when(pl.col('shotDistance').is_null()).then(pl.col('SHOT_DISTANCE')).otherwise(pl.col('shotDistance')).alias('shotDistance')
)

# Convert score to be relative to player taking actions perspective
df_merged = df_merged.with_columns(
    pl.when(pl.col('teamTricode') == pl.col('HTM'))
        .then(pl.col('score_margin_after') - pl.col('score_change'))
    .otherwise(pl.col('score_margin_after')*-1 - pl.col('score_change')).alias('relative_margin_before')
)

In [ ]:
df_merged.head(5)

In [ ]:
# Confirm no mismatching data between event types & shot flags
made_shots_mismatch = df_merged.filter(
    (pl.col('EVENT_TYPE') =='Made Shot') & (pl.col('SHOT_ATTEMPTED_FLAG')!=1) & (pl.col('SHOT_MADE_FLAG')!=1)
)

missed_shots_mismatch = df_merged.filter(
    (pl.col('EVENT_TYPE') =='Missed Shot') & (pl.col('SHOT_ATTEMPTED_FLAG')!=1) & (pl.col('SHOT_MADE_FLAG')!=0)
)

In [ ]:
# Drop redundant (i.e. duplicate) columns
df_combined = df_merged.drop(['period', 'personId', 'actionType', 'GAME_DATE', 'SHOT_DISTANCE', 'teamId'])

# Drop columns deemed not required
df_combined = df_combined.drop([
    'EVENT_TYPE',
    'MINUTES_REMAINING',
    'SECONDS_REMAINING',
    'foulPersonalTotal',
    'foulTechnicalTotal',
    'foulDrawnPlayerName',
    'foulDrawnPersonId',
    'area',
    'areaDetail',
    'shotResult',
    'officialId',
    'turnoverTotal',
    'stealPlayerName',
    'stealPersonId',
    'reboundTotal',
    'reboundDefensiveTotal',
    'reboundOffensiveTotal',
    'blockPlayerName',
    'blockPersonId',
    'jumpBallLostPlayerName',
    'jumpBallLostPersonId',
    'jumpBallWonPlayerName',
    'jumpBallWonPersonId', 
    'jumpBallRecoveredName',    
    'jumpBallRecoverdPersonId',
    'shotActionNumber', 
    'SHOT_ATTEMPTED_FLAG', 
    'playerName',
    'playerNameI', 
    'teamTricode', 
    'possession',
    'xLegacy',
    'yLegacy',
    'x',
    'y',
    'description',
    'personIdsFilter',
    'descriptor'
])
df_combined

## Missing Data

Deal with null, NaN data

In [ ]:
# Replace nulls in pointsTotal & shotDistance with 0 - effectively what null means in this column
df_combined = df_combined.with_columns(
    pl.col('pointsTotal').fill_null(0).alias('pointsTotal')
)

Drop null values

In [ ]:
# Drop records with null values for scoreAway or ScoreHome
df_combined = df_combined.drop_nulls('score_margin_after')

df_nulls = df_combined.filter(pl.col('relative_margin_before').is_null() | pl.col('LOC_Y').is_null())
test = df_combined.filter(pl.col('relative_margin_before') == 0)

df_nulls

In [ ]:
df_combined.columns

In [ ]:
# Subset of numeric columns for correlation
numerical = [
    'PERIOD',
    'LOC_X',
    'LOC_Y',
    'pointsTotal',
    'relative_margin_before',
    'shotDistance',
    'SHOT_MADE_FLAG',
    'time_remaining'
]

# Subset of columns including categorical variables only
categorical = [
    'ACTION_TYPE',
    'isHomeTeam',
    'PLAYER_ID',
    'SHOT_ZONE_BASIC',
    'SHOT_ZONE_AREA',
    'SHOT_ZONE_RANGE',
    'subType',
    'SHOT_MADE_FLAG',
    'time_remaining'
]
    
# Subset of columns to be used in classifier models
classification = [   
    'PERIOD',
    'LOC_X',
    'LOC_Y',
    'pointsTotal',
    'relative_margin_before',
    'shotDistance',
    'SHOT_MADE_FLAG',
    'time_remaining'
]

df_numerical = df_combined.select(numerical)
df_categorical = df_combined.select(categorical)
df_model = df_combined.select(classification)
df_model


In [ ]:
df_categorical

## Encoding

Encode categorical variables to enable use in classifiers

In [ ]:
df_encoded = df_combined.select(classification).to_dummies(
    ['subType',
     'side',
     'SHOT_ZONE_BASIC',
     'SHOT_ZONE_AREA',
     'SHOT_ZONE_RANGE'
     ]
)
df_encoded

In [ ]:
df_encoded = df_encoded.to_pandas()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(10, 8))


sns.kdeplot(
    x=df_encoded["LOC_X"], 
    y=df_encoded["LOC_Y"], 
    shade=True, 
    cmap="Reds", 
    fill=True, 
    thresh=0.05, 
    levels=100
)


sns.scatterplot(
    x="LOC_X", 
    y="LOC_Y", 
    hue="SHOT_MADE_FLAG", 
    data=df_encoded.sample(2000), 
    palette={0:"blue", 1:"green"},
    alpha=0.5,
    s=10
)

plt.title("Shot Heatmap and Distribution")
plt.xlabel("LOC_X (Court Position)")
plt.ylabel("LOC_Y (Court Position)")
plt.legend(title="Shot Made", loc="upper right")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(data=df_encoded, x="shotDistance", hue="SHOT_MADE_FLAG", bins=30, kde=True, stat="density", common_norm=False)
plt.title("Distribution of Shot Distance by Shot Result")
plt.xlabel("Shot Distance")
plt.ylabel("Density")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

X = df_encoded[["shotDistance"]]
y = df_encoded["SHOT_MADE_FLAG"]


model = LogisticRegression()
model.fit(X, y)


x_range = np.linspace(X.min()[0], X.max()[0], 300).reshape(-1, 1)
y_pred = model.predict_proba(x_range)[:, 1]

plt.figure(figsize=(10,6))


plt.scatter(X.sample(2000, random_state=42), 
            y.sample(2000, random_state=42), 
            alpha=0.05, color="gray", label="Shots (sample)")

plt.plot(x_range, y_pred, color="red", linewidth=2, label="Logistic Regression Curve")

plt.xlabel("Shot Distance")
plt.ylabel("Probability of Made Shot")
plt.title("Logistic Regression: Shot Distance vs Make Probability")
plt.legend()
plt.show()

In [ ]:
from sklearn.tree import DecisionTreeClassifier
tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X, y)

# 生成距离范围
x_range = np.linspace(X.min()[0], X.max()[0], 300).reshape(-1, 1)
y_pred_tree = tree.predict_proba(x_range)[:,1]

# 画图
plt.figure(figsize=(10,6))
plt.scatter(X.sample(2000, random_state=42), y.sample(2000, random_state=42),
            alpha=0.05, color="gray", label="Shots (sample)")
plt.plot(x_range, y_pred_tree, color="blue", linewidth=2, label="Decision Tree Curve")
plt.xlabel("Shot Distance")
plt.ylabel("Probability of Made Shot")
plt.title("Decision Tree: Shot Distance vs Make Probability")
plt.legend()
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 随机森林模型
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X, y)

# 预测
y_pred_rf = rf.predict_proba(x_range)[:,1]

# 画图
plt.figure(figsize=(10,6))
plt.scatter(X.sample(2000, random_state=42), y.sample(2000, random_state=42),
            alpha=0.05, color="gray", label="Shots (sample)")
plt.plot(x_range, y_pred_rf, color="green", linewidth=2, label="Random Forest Curve")
plt.xlabel("Shot Distance")
plt.ylabel("Probability of Made Shot")
plt.title("Random Forest: Shot Distance vs Make Probability")
plt.legend()
plt.show()

In [ ]:
import xgboost as xgb
dtrain = xgb.DMatrix(X, label=y,feature_names=["shotDistance"])

# 训练参数
params = {
    "objective": "binary:logistic",  # 二分类
    "eval_metric": "logloss",
    "max_depth": 6,
    "eta": 0.1,
    "seed": 42
}

# 训练 XGBoost 模型
xgb_model = xgb.train(params, dtrain, num_boost_round=200)

# 生成距离范围
x_range = np.linspace(X.min()[0], X.max()[0], 300).reshape(-1, 1)
dtest = xgb.DMatrix(x_range,feature_names=["shotDistance"])
y_pred_xgb = xgb_model.predict(dtest)

# 画图
plt.figure(figsize=(10,6))
plt.scatter(X.sample(2000, random_state=42), y.sample(2000, random_state=42),
            alpha=0.05, color="gray", label="Shots (sample)")
plt.plot(x_range, y_pred_xgb, color="orange", linewidth=2, label="XGBoost Curve")
plt.xlabel("Shot Distance")
plt.ylabel("Probability of Made Shot")
plt.title("XGBoost: Shot Distance vs Make Probability")
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, log_loss, roc_curve
log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)
y_pred_log = log_reg.predict(X_test)
y_pred_log_proba = log_reg.predict_proba(X_test)[:,1]

tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)
y_pred_tree_proba = tree.predict_proba(X_test)[:,1]

rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_pred_rf_proba = rf.predict_proba(X_test)[:,1]

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)
params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "max_depth": 6,
    "eta": 0.1,
    "seed": 42
}
xgb_model = xgb.train(params, dtrain, num_boost_round=200)
y_pred_xgb_proba = xgb_model.predict(dtest)
y_pred_xgb = (y_pred_xgb_proba > 0.5).astype(int)


results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest", "XGBoost"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_tree),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ],
    "AUC": [
        roc_auc_score(y_test, y_pred_log_proba),
        roc_auc_score(y_test, y_pred_tree_proba),
        roc_auc_score(y_test, y_pred_rf_proba),
        roc_auc_score(y_test, y_pred_xgb_proba)
    ],
    "F1": [
        f1_score(y_test, y_pred_log),
        f1_score(y_test, y_pred_tree),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb)
    ],
    "LogLoss": [
        log_loss(y_test, y_pred_log_proba),
        log_loss(y_test, y_pred_tree_proba),
        log_loss(y_test, y_pred_rf_proba),
        log_loss(y_test, y_pred_xgb_proba)
    ]
})

print(results)


plt.figure(figsize=(10,8))

# Logistic
fpr, tpr, _ = roc_curve(y_test, y_pred_log_proba)
plt.plot(fpr, tpr, label=f"Logistic (AUC={roc_auc_score(y_test, y_pred_log_proba):.3f})")

# Tree
fpr, tpr, _ = roc_curve(y_test, y_pred_tree_proba)
plt.plot(fpr, tpr, label=f"Decision Tree (AUC={roc_auc_score(y_test, y_pred_tree_proba):.3f})")

# RF
fpr, tpr, _ = roc_curve(y_test, y_pred_rf_proba)
plt.plot(fpr, tpr, label=f"Random Forest (AUC={roc_auc_score(y_test, y_pred_rf_proba):.3f})")

# XGB
fpr, tpr, _ = roc_curve(y_test, y_pred_xgb_proba)
plt.plot(fpr, tpr, label=f"XGBoost (AUC={roc_auc_score(y_test, y_pred_xgb_proba):.3f})")

plt.plot([0,1], [0,1], "k--")  # baseline
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves Comparison")
plt.legend()
plt.show()

In [ ]:
df_categorical = df_categorical.to_pandas()


In [ ]:
df_categorical.head()

In [ ]:
df_categorical["time_remaining_sec"] = df_categorical["time_remaining"].dt.total_seconds()

In [ ]:
clutch_df = df_categorical[df_categorical["time_remaining_sec"] <= 120]
clutch_df

In [ ]:
clutch_df["SHOT_ZONE_BASIC"].unique()

In [ ]:
df_combined_pd = df_combined.to_pandas()  

clutch_df["shotDistance"] = df_combined_pd .loc[clutch_df.index, "shotDistance"]
clutch_df["SHOT_TYPE"]   = df_combined_pd .loc[clutch_df.index, "SHOT_TYPE"]


In [ ]:
clutch_stats_all = (
    clutch_df.groupby("subType")
    .agg(
        attempts=("SHOT_MADE_FLAG", "count"),
        makes=("SHOT_MADE_FLAG", "sum")
    )
    .reset_index()
)

clutch_stats_all["fg_pct"] = clutch_stats_all["makes"] / clutch_stats_all["attempts"]
clutch_stats_all

In [ ]:

two_pt_zones = [
    "Mid-Range", "Restricted Area", "In The Paint (Non-RA)", "Backcourt"
]
three_pt_zones = [
    "Above the Break 3", "Left Corner 3", "Right Corner 3"
]

clutch_df["shot_type"] = clutch_df["SHOT_ZONE_BASIC"].apply(
    lambda x: "3PT" if x in three_pt_zones else "2PT"
)

In [ ]:
fg_stats = (
    clutch_df.groupby("shot_type")["SHOT_MADE_FLAG"]
    .agg(["count", "sum", "mean"])
    .rename(columns={"count": "attempts", "sum": "makes", "mean": "fg_pct"})
)
print(fg_stats)